In [1]:
import requests
import json
import pprint
import pandas as pd
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/74.0.3729.169 YaBrowser/19.6.1.153 Yowser/2.5 Safari/537.36'}

In [2]:
df_markets = pd.read_csv("tyler_diddy_polymarkets.csv", sep=";")
df_markets.columns

Index(['Unnamed: 0', 'id', 'slug', 'conditionId', 'liquidity', 'volume',
       'open_interest', 'volume24hr', 'volume1wk', 'volume1mo', 'volume1yr',
       'price_yes', 'price_no', 'oneWeekPriceChange'],
      dtype='str')

In [3]:
df_markets["conditionId"][0]

'0xdfea8851512360fb70c4b4ed29ed369b6dbcdfc58b2493b66cba65faab622129'

In [4]:
#Let's try calling the API first to know what info we get
h_url = "https://data-api.polymarket.com/holders"
params = {
    "market": df_markets["conditionId"][0]
}
response = requests.get(h_url, headers=headers, params=params)
test = response.json()
test[0]

{'token': '110121849054524311559637722508594302357291606532161889048119752094089952931603',
 'holders': [{'proxyWallet': '0xa7a6cd5399040aa661ee595f4421337d80188117',
   'bio': '',
   'asset': '110121849054524311559637722508594302357291606532161889048119752094089952931603',
   'pseudonym': 'Unsteady-Drake',
   'amount': 10515.915772,
   'displayUsernamePublic': True,
   'outcomeIndex': 0,
   'name': 'KoloJones',
   'profileImage': '',
   'profileImageOptimized': '',
   'verified': False},
  {'proxyWallet': '0xf44dcf50490507a44a6244dc07f50686e71060af',
   'bio': '',
   'asset': '110121849054524311559637722508594302357291606532161889048119752094089952931603',
   'pseudonym': 'Recent-Eveningwear',
   'amount': 6140.58,
   'displayUsernamePublic': True,
   'outcomeIndex': 0,
   'name': 'Certify0148',
   'profileImage': '',
   'profileImageOptimized': '',
   'verified': False},
  {'proxyWallet': '0x44fccfc4f7653fc79c386d1bcb2181d730a4ac29',
   'bio': '',
   'asset': '11012184905452431155963

In [5]:
test[1]["holders"]
#test[0] contains yes-holders and test[1] contains no-holders

[{'proxyWallet': '0xe4de861d59e174d3d153a5968bc122c66f30c949',
  'bio': '',
  'asset': '55924008678331392948336312806293360142875154524733691603322942979182976457930',
  'pseudonym': 'Firsthand-Offense',
  'amount': 10000,
  'displayUsernamePublic': True,
  'outcomeIndex': 1,
  'name': 'On.The.Spectrum',
  'profileImage': '',
  'profileImageOptimized': '',
  'verified': False},
 {'proxyWallet': '0x00d1bfdb216e363bdb1a04dc02865a4dd99ec8a4',
  'bio': '',
  'asset': '55924008678331392948336312806293360142875154524733691603322942979182976457930',
  'pseudonym': 'Any-Adviser',
  'amount': 5987.255074,
  'displayUsernamePublic': True,
  'outcomeIndex': 1,
  'name': 'placerv2',
  'profileImage': '',
  'profileImageOptimized': '',
  'verified': False},
 {'proxyWallet': '0x96f795d0821b75a1c7f886b2c1cd49d108b7d6ae',
  'bio': '',
  'asset': '55924008678331392948336312806293360142875154524733691603322942979182976457930',
  'pseudonym': 'Weekly-Accounting',
  'amount': 5152.95095,
  'displayUsernam

In [6]:
#Calling the API to get top holders of both markets
whale_master_list = []

for index, row in df_markets.iterrows():
    cond_id = row["conditionId"]
    market_title = row["slug"]

    h_url = f"https://data-api.polymarket.com/holders"
    params = {
        "market": cond_id,
        "limit": 100,
        "offset": 0,
    }
    response = requests.get(h_url, headers=headers, params=params)
    raw_holders = response.json()

    #THE API returns the yes holders first, so we set i == 0 to yes.
    for i, item in enumerate(raw_holders):
        if "holders" not in item:
            continue
    
        side = "YES" if i == 0 else "NO"
        
        #getting the holders data
        for holder in item["holders"]:
            whale_master_list.append(
                {
                "side": side,
                "market": market_title,
                "username": holder.get("pseudonym"),
                "wallet": holder.get("proxyWallet"),
                "shares": holder.get("amount"),
                "yes_no_index": holder.get("outcomeIndex"),
            }
            )
                
df_master_whales = pd.DataFrame(whale_master_list)    

In [7]:
#Double checking shows that these were not actually usernames
df_master_whales["pseudonym"] = df_master_whales["username"]
df_master_whales = df_master_whales.drop(columns=["username"])

In [8]:
df_master_whales.to_csv("all_whales.csv")

In [9]:
df_master_whales

,side,market,wallet,shares,yes_no_index,pseudonym
0,YES,tyler-robinson-convicted-of-homicide,0xa7a6cd5399040aa661ee595f4421337d80188117,10515.915772,0,Unsteady-Drake
1,YES,tyler-robinson-convicted-of-homicide,0xf44dcf50490507a44a6244dc07f50686e71060af,6140.580000,0,Recent-Eveningwear
2,YES,tyler-robinson-convicted-of-homicide,0x44fccfc4f7653fc79c386d1bcb2181d730a4ac29,4881.270000,0,Clean-Windscreen
3,YES,tyler-robinson-convicted-of-homicide,0xc8ab97a9089a9ff7e6ef0688e6e591a066946418,3294.142968,0,Hideous-Racer
4,YES,tyler-robinson-convicted-of-homicide,0xfc1490c043e5ed6ec2ce7ea003e6b540b0eb2e9b,3190.090000,0,Bountiful-Clinic
...,...,...,...,...,...,...
305,NO,diddy-found-guilty-of-sex-trafficking,0xfe940ab2a3ad0b6e2295afb2b4c07f871369f965,5.000000,1,Straight-Ruler
306,NO,diddy-found-guilty-of-sex-trafficking,0x3774515d6f0e682434cf176886c1f4c0fc8478c4,3.448274,1,Aged-Suspect
307,NO,diddy-found-guilty-of-sex-trafficking,0x7bc927893bd89b981adfa4200b703a36958da102,2.857141,1,Full-Minimalism
308,NO,diddy-found-guilty-of-sex-trafficking,0xf7421c80e6212ae5127e74b932cc310b7e31be2d,2.857141,1,These-River


In [10]:
#Now we create a list of the top holders from each market
df_master_whales
df_tyler_filtered = df_master_whales[df_master_whales["market"] == "tyler-robinson-convicted-of-homicide"]

In [11]:
df_tyler_filtered

,side,market,wallet,shares,yes_no_index,pseudonym
0,YES,tyler-robinson-convicted-of-homicide,0xa7a6cd5399040aa661ee595f4421337d80188117,10515.915772,0,Unsteady-Drake
1,YES,tyler-robinson-convicted-of-homicide,0xf44dcf50490507a44a6244dc07f50686e71060af,6140.580000,0,Recent-Eveningwear
2,YES,tyler-robinson-convicted-of-homicide,0x44fccfc4f7653fc79c386d1bcb2181d730a4ac29,4881.270000,0,Clean-Windscreen
3,YES,tyler-robinson-convicted-of-homicide,0xc8ab97a9089a9ff7e6ef0688e6e591a066946418,3294.142968,0,Hideous-Racer
4,YES,tyler-robinson-convicted-of-homicide,0xfc1490c043e5ed6ec2ce7ea003e6b540b0eb2e9b,3190.090000,0,Bountiful-Clinic
...,...,...,...,...,...,...
195,NO,tyler-robinson-convicted-of-homicide,0x57b9acdde2dff37c39ad37f24f3293c8d2330464,5.447760,1,
196,NO,tyler-robinson-convicted-of-homicide,0xe6000afcee6ea70d272e3ca3517708add214a907,5.343282,1,
197,NO,tyler-robinson-convicted-of-homicide,0xe7505c94c7d5980919ad83b8e614142891094850,5.222221,1,
198,NO,tyler-robinson-convicted-of-homicide,0xcc1a5d9160199b0862fbe0870d211974a8d87044,5.149252,1,


In [12]:
#We do the same with the other market
df_master_whales
df_diddy_filtered = df_master_whales[df_master_whales["market"] == "diddy-found-guilty-of-sex-trafficking"]

In [13]:
len(df_diddy_filtered)

110

In [14]:
count_diddy_yes = (df_diddy_filtered["side"] == "YES").sum()
count_tyler_yes = (df_tyler_filtered["side"] == "YES").sum()
count_diddy_no = (df_diddy_filtered["side"] == "NO").sum()
count_tyler_no = (df_tyler_filtered["side"] == "NO").sum()
print(count_diddy_yes, count_tyler_yes, count_diddy_no, count_tyler_no)

100 100 10 100


In [15]:
#Double checking with the polymarket website it seems to be true that there were only 10 "no" holders for the diddy conviction market.

In [16]:
#Let's round the shares and then analyze the number of shares in each market
df_diddy_filtered["shares"] = df_diddy_filtered["shares"].round()
df_tyler_filtered["shares"] = df_tyler_filtered["shares"].round()
#Since there is an unequal amount of holders in these dfs we have to create new subsets.
df_tyler_yes = df_tyler_filtered[df_tyler_filtered["side"] == "YES"]
df_tyler_no = df_tyler_filtered[df_tyler_filtered["side"] == "NO"]
df_diddy_yes = df_diddy_filtered[df_diddy_filtered["side"] == "YES"]
df_diddy_no = df_diddy_filtered[df_diddy_filtered["side"] == "NO"]

In [17]:
import matplotlib.pyplot as plt

In [18]:
#Adding up the total shares kept by the top 100 holders on the leading side in both markets
tot_leading_shares = (df_tyler_no["shares"].sum()) + (df_diddy_yes["shares"].sum())
print(f"The 100 biggest yes-holders for the diddy-market and 100 biggest no-holders for Robinson together held {tot_leading_shares} shares.")

The 100 biggest yes-holders for the diddy-market and 100 biggest no-holders for Robinson together held 242728.0 shares.


In [19]:
#Since we know the volume of both markets (which is the same as shares traded)
#We can check how much of the total market is with the 100 biggest holders
tyler_vol = 177352
diddy_vol = 797961
df_tyler_filtered = df_tyler_filtered.sort_values("shares", ascending=False)
df_diddy_filtered = df_diddy_filtered.sort_values("shares", ascending=False)
df_diddy_filtered.head(10)

,side,market,wallet,shares,yes_no_index,pseudonym
200,YES,diddy-found-guilty-of-sex-trafficking,0xc1b02266e94b87473cd4e38a61f13c69bf5a4535,131519.0,0,Squiggly-Bonding
201,YES,diddy-found-guilty-of-sex-trafficking,0x2e7f06e8b568b9a04e152a549500e6a54f99b1d4,36258.0,0,
202,YES,diddy-found-guilty-of-sex-trafficking,0x65bf3bfd2cd0506e8e41752a01e7af7817c94b91,6000.0,0,Masculine-Pinot
203,YES,diddy-found-guilty-of-sex-trafficking,0x157fa171d15d3b9f9783a0c3f8700c95a12b5b27,4386.0,0,
204,YES,diddy-found-guilty-of-sex-trafficking,0xcf7c3d0c796b2aaaccfd613236431a0ec7cdf2cb,3000.0,0,Menacing-Hope
205,YES,diddy-found-guilty-of-sex-trafficking,0x7fda167dcaa5ffe2c6c34791c103c4bfcee3e430,2600.0,0,Rigid-Weight
206,YES,diddy-found-guilty-of-sex-trafficking,0xf6d8e74e76e79dcc7061c20824113cec164114ab,1504.0,0,Whopping-Manufacturing
207,YES,diddy-found-guilty-of-sex-trafficking,0xccdac3c7b285bb9402bee9e0e58fe76480af4937,1243.0,0,Overcooked-Diaphragm
208,YES,diddy-found-guilty-of-sex-trafficking,0x00fa1da7d2b8baa703a1d3554b5d1338be9030be,857.0,0,Delirious-Buzzard
209,YES,diddy-found-guilty-of-sex-trafficking,0x477a7d28ed844984088129207c0afcb2459e2d8e,450.0,0,Fair-Record


In [20]:
print(f"The top 100 holders of the Tyler market holds {((df_tyler_filtered["shares"].head(100).sum()) / 177352) * 100} percent of the total market.")
print(f"The top 100 holders of the Diddy market held {((df_diddy_filtered["shares"].head(100).sum()) / 797961) * 100} percent of the total market.")

The top 100 holders of the Tyler market holds 55.97625062023547 percent of the total market.
The top 100 holders of the Diddy market held 24.05430841858186 percent of the total market.


In [21]:
print(f"The top 10 holders of the Tyler market holds {((df_tyler_filtered["shares"].head(10).sum()) / 177352) * 100} percent of the total market.")
print(f"The top 10 holders of the Diddy market held {((df_diddy_filtered["shares"].head(10).sum()) / 797961) * 100} percent of the total market.")

The top 10 holders of the Tyler market holds 31.576187468988227 percent of the total market.
The top 10 holders of the Diddy market held 23.537115222423154 percent of the total market.


In [22]:
#The Tyler Robinson market is more evenly spread out.
#Maybe that says something about the increasing mainstream-ness of prediction markets.

In [23]:
market_holdings = {
    "market": ["tyler_market", "diddy_market"],
    "total_shares": [177352, 797961],
    "top_100_shares": [df_tyler_filtered["shares"].head(100).sum(), df_diddy_filtered["shares"].head(100).sum()],
    "top_100_perc": [((df_tyler_filtered["shares"].head(100).sum()) / 177352) * 100, ((df_diddy_filtered["shares"].head(100).sum()) / 797961) * 100],
    "top_10_shares": [df_tyler_filtered["shares"].head(10).sum(), df_diddy_filtered["shares"].head(10).sum()],
    "top_10_perc": [((df_tyler_filtered["shares"].head(10).sum()) / 177352) * 100, ((df_diddy_filtered["shares"].head(10).sum()) / 797961) * 100],
    "top_diff": [df_tyler_filtered["shares"].head(100).sum() - df_tyler_filtered["shares"].head(10).sum(), df_diddy_filtered["shares"].head(100).sum() - df_diddy_filtered["shares"].head(10).sum()],
    "top_diff_perc": [((df_tyler_filtered["shares"].head(100).sum()) / 177352) * 100 - ((df_tyler_filtered["shares"].head(10).sum()) / 177352) * 100,
                      ((df_diddy_filtered["shares"].head(100).sum()) / 797961) * 100 - ((df_diddy_filtered["shares"].head(10).sum()) / 797961) * 100],
}

In [24]:
market_holdings = pd.DataFrame(market_holdings)
#I realized that the API was giving me bad data for the diddy_market so we are removing it from the df.
market_holdings = market_holdings[market_holdings["market"] != "diddy_market"]
market_holdings

,market,total_shares,top_100_shares,top_100_perc,top_10_shares,top_10_perc,top_diff,top_diff_perc
0,tyler_market,177352,99275.0,55.976251,56001.0,31.576187,43274.0,24.400063


In [25]:
market_holdings_chart = market_holdings.drop(columns=["total_shares", "market", "top_100_shares", "top_10_shares", "top_diff", "top_diff_perc"])

In [26]:
market_holdings_chart["top_100_perc"] = (market_holdings_chart["top_100_perc"]) - (market_holdings_chart["top_10_perc"])
market_holdings_chart["rest"] = 100 - ((market_holdings_chart["top_100_perc"]) + (market_holdings_chart["top_10_perc"]))
market_holdings_chart

,top_100_perc,top_10_perc,rest
0,24.400063,31.576187,44.023749


In [27]:
market_holdings_chart.to_csv("market_holdings_chart_tyler_diddy.csv")

In [28]:
df_diddy_filtered.to_csv("diddy_holders_filtered.csv")
df_tyler_filtered.to_csv("tyler_holders_filtered.csv")

In [29]:
#let's see if there are people who traded both markets
tyler_wallets = set(df_tyler_filtered["wallet"].astype(str).str.lower().str.strip())
diddy_wallets = set(df_diddy_filtered["wallet"].astype(str).str.lower().str.strip())
overlap = diddy_wallets.intersection(tyler_wallets)
len(overlap)

0

In [30]:
#We double check with another method that there really isn't any overlapping traders
df_overlap = pd.merge(
    df_diddy_filtered, df_tyler_filtered, on="wallet", suffixes=("_diddy", "_tyler")
)

# Clean up the output to see the most important comparison columns
comparison_view = df_overlap[
    [
        "wallet",
        "pseudonym_diddy",
        "shares_diddy",
        "side_diddy",
        "pseudonym_tyler",
        "shares_tyler",
        "side_tyler",
    ]
]
print(f"Found {len(comparison_view)} overlapping traders!")
print(comparison_view.head())

Found 0 overlapping traders!
Empty DataFrame
Columns: [wallet, pseudonym_diddy, shares_diddy, side_diddy, pseudonym_tyler, shares_tyler, side_tyler]
Index: []


In [31]:
df_top_ten = pd.concat([df_tyler_filtered.head(10), df_diddy_filtered.head(10)], ignore_index=True)
df_top_ten.to_csv("top_ten_whales_both.csv")